# Raw data checks before mapping (Goodreads, DBpedia, Amazon)

WDI Phase I, Group 6.

Before mapping the three datasets to the integrated schema (Table 2 of our proposal) in MapForce, we profile the raw files in Python. The goal is to find out what is wrong or inconsistent in the data, so that every rule in the mappings has a reason. Each check below maps to one decision in MapForce, summarised in section 5.

Target attributes: `title`, `authors` (list), `genres` (list), `publication_date` (date), `publisher`, `page_count` (int), `language`, `format`, `edition`, `rating` (float), `ratings_count` (int), `price` (float), `series`, `awards` (list) and `isbn`. The ISBN is only an identifier for the overlap estimation and the gold standard, we do not match on it.

To run it, execute the two cells in section 0 (check the paths first), then any dataset section. The datasets are analysed separately.

## 0. Setup

The first cell holds the file paths, the second one the checking functions.

In [ ]:
import csv
import os
import re
import sys
import xml.etree.ElementTree as ET
from collections import Counter

csv.field_size_limit(10**9)

PATHS = {
    "Goodreads": r"C:\Users\safas\Downloads\books_1.Best_Books_Ever(2).csv",
    "DBpedia": r"C:\Users\safas\Downloads\dbpedia_book_data.xml",
    "Amazon": r"C:\Users\safas\Downloads\amazon_books_subset.csv",
}


In [ ]:
# which column / element is what (names as in Table 2 of the proposal)
CFG = {
    "Goodreads": dict(dates=["publishDate"], numbers=["price", "pages", "rating", "numRatings"],
                      isbn=["isbn"], lists=["genres", "awards"], authors="author",
                      text=["title", "publisher", "genres", "author"]),
    "DBpedia": dict(dates=["release_date", "year"], numbers=["pages", "avg_star_rating", "number_of_reviews"],
                    isbn=["isbns"], lists=["genres", "authors"], authors=None,
                    text=["title", "publisher", "genres", "authors"]),
    "Amazon": dict(dates=["publication_date"], numbers=["pages", "average_rating", "ratings_count", "price"],
                   isbn=["isbn13", "isbn10"], lists=["genres", "authors"], authors=None,
                   text=["title", "publisher", "genres", "authors"]),
}

MONTH = (r"(Jan(uary)?|Feb(ruary)?|Mar(ch)?|Apr(il)?|May|Jun(e)?|Jul(y)?|Aug(ust)?|"
         r"Sep(t(ember)?)?|Oct(ober)?|Nov(ember)?|Dec(ember)?)")

# class labels (used as keys below)
YEAR_ONLY = "year only (-> YYYY-01-01)"
REAL_JAN1 = "real 1 January"
FULL_DATE = "full date (day known)"
MONTH_YEAR = "month + year (no day)"
NOT_A_DATE = "NOT A DATE / other format"


# ------------------------------- classification -------------------------------
def short(v, n=40):
    return re.sub(r"\s+", " ", str(v)).strip()[:n]


def date_class(v):
    v = v.strip()
    if not v:
        return "empty"
    if re.fullmatch(r"\d{4}", v):
        return YEAR_ONLY
    if re.fullmatch(r"01/01/(\d\d|\d{4})|January 1(st)?,? \d{4}|\d{4}-01-01", v):
        return REAL_JAN1
    if re.fullmatch(r"\d\d/\d\d/(\d\d|\d{4})|" + MONTH + r"\.? \d{1,2}(st|nd|rd|th)?,? \d{4}|\d{4}-\d\d-\d\d", v):
        return FULL_DATE
    if re.fullmatch(MONTH + r"\.? \d{4}", v):
        return MONTH_YEAR
    return NOT_A_DATE


def number_class(col, v):
    v = v.strip()
    if not v:
        return "empty"
    if re.fullmatch(r"\d+(\.\d+)?", v):
        return "valid"
    if col == "pages" and re.fullmatch(r"\d+\s*pages?", v, re.I):
        return "text 'N page(s)' (-> N)"
    return "invalid (dropped)"


def isbn_class(v):
    v = v.strip()
    if not v:
        return "empty"
    if v == "9999999999999":
        return "placeholder 9999999999999"
    if re.search(r"[Ee]\+\d+$", v):
        return "exponent format (Excel damage)"
    plain = v.replace("-", "").replace(" ", "")
    if re.fullmatch(r"\d{13}", plain):
        return "valid ISBN-13"
    if re.fullmatch(r"\d{9}[\dXx]", plain):
        return "valid ISBN-10"
    if re.fullmatch(r"B0[0-9A-Z]{8}", v):
        return "ASIN (not an ISBN)"
    return "other / junk"


def list_class(v):
    v = v.strip()
    if not v:
        return "empty"
    if v == "[]":
        return "empty list []"
    if v.startswith("['"):
        return "Python list ['A', 'B']"
    if v.startswith('["'):
        if re.search(r"\",\s*'|',\s*\"|\", \"", v):
            return "Python list with apostrophes (mixed quotes)"
        return 'JSON list ["A","B"]'
    if v.startswith("["):
        return "list, other format"
    return "plain text (no list format)"


# ----------------------------------- reading ----------------------------------
LIST_TAGS_XML = {"isbns", "authors", "genres"}


def read_csv(path):
    f = open(path, newline="", encoding="utf-8")
    r = csv.reader(f)
    header = next(r)

    def gen():
        for nr, row in enumerate(r, start=2):          # row 1 is the header
            yield nr, (dict(zip(header, row)) if len(row) == len(header) else None)
    return gen()


def read_xml(path):
    def gen():
        nr = 0
        for _, el in ET.iterparse(path, events=("end",)):
            if el.tag != "book":
                continue
            nr += 1
            rec = {}
            for k in el:
                if len(k) or k.tag in LIST_TAGS_XML:
                    rec[k.tag] = [(x.text or "").strip() for x in k]
                else:
                    rec[k.tag] = (k.text or "").strip()
            yield nr, rec
            el.clear()
    return gen()


# ----------------------------------- output -----------------------------------
def remember(examples, key, v):
    examples.setdefault(key, [])
    if len(examples[key]) < 3:
        examples[key].append(short(v, 28))


def show(title, counter, total, examples=None):
    print(f"\n  {title}")
    for k, n in counter.most_common():
        line = f"    {k:<46}{n:>8}{n / max(total, 1):>8.1%}"
        if examples and examples.get(k):
            line += f"   {examples[k]}"
        print(line)


def is_empty(v):
    return v == [] or (isinstance(v, str) and v.strip() in ("", "[]"))


# ---------------------------------- analysis ----------------------------------
def analyse(name, cfg, path):
    print(f"{name}   ({path})")
    print("=" * 86)
    gen = read_xml(path) if path.lower().endswith(".xml") else read_csv(path)

    total = shifted = 0
    dates = {c: Counter() for c in cfg["dates"]}
    d_ex = {c: {} for c in cfg["dates"]}
    d_junk = {c: [] for c in cfg["dates"]}
    nums = {c: Counter() for c in cfg["numbers"]}
    n_ex = {c: {} for c in cfg["numbers"]}
    isbns = {c: Counter() for c in cfg["isbn"]}
    i_ex = {c: {} for c in cfg["isbn"]}
    lists = {c: Counter() for c in cfg["lists"]}
    l_ex = {c: {} for c in cfg["lists"]}
    filled = Counter()
    roles, with_role, with_amp, extra = Counter(), 0, 0, Counter()

    for nr, rec in gen:
        total += 1
        if rec is None:
            shifted += 1
            continue
        for k, v in rec.items():
            if not is_empty(v):
                filled[k] += 1
        for c in cfg["dates"]:
            v = rec.get(c, "")
            k = date_class(v)
            dates[c][k] += 1
            remember(d_ex[c], k, v)
            if k == NOT_A_DATE and len(d_junk[c]) < 10:
                d_junk[c].append((nr, short(v, 60)))
        for c in cfg["numbers"]:
            v = rec.get(c, "")
            k = number_class(c, v)
            nums[c][k] += 1
            if k != "valid":
                remember(n_ex[c], k, v)
        for c in cfg["isbn"]:
            vals = rec.get(c, "")
            for v in (vals if isinstance(vals, list) else [vals]):
                k = isbn_class(v)
                isbns[c][k] += 1
                remember(i_ex[c], k, v)
        for c in cfg["lists"]:
            v = rec.get(c, "")
            k = ("list (XML elements)" if v else "empty") if isinstance(v, list) else list_class(v)
            lists[c][k] += 1
            remember(l_ex[c], k, v)
        if cfg["authors"]:
            r = re.findall(r"\(([^)]*)\)", rec.get(cfg["authors"], ""))
            if r:
                with_role += 1
                roles.update(r)
        text = " ".join(" ".join(v) if isinstance(v, list) else v
                        for v in (rec.get(c, "") for c in cfg["text"]))
        if "&" in text:
            with_amp += 1
        dataset_checks(name, rec, extra)

    ok = total - shifted
    print(f"\n[1] STRUCTURE\n    books/rows in total: {total}")
    print(f"    rows with wrong column count (skipped in this analysis): {shifted}")

    print("\n[2] DATES  (target: xs:date, YYYY-MM-DD)")
    for c in cfg["dates"]:
        show(f"column '{c}'", dates[c], ok, d_ex[c])
        if d_junk[c]:
            print(f"    examples of NOT A DATE (record, value): {d_junk[c]}")
    c = cfg["dates"][0]
    added, real = dates[c][YEAR_ONLY], dates[c][REAL_JAN1]
    print(f"\n    IS -01-01 UNIQUE in '{c}'?")
    print(f"      dates ending in 01-01 in the target: {added + real}")
    if added + real:
        print(f"      added by us (year only):          {added} ({added / (added + real):.1%})")
        print(f"      real 1 January in the raw data:   {real} ({real / (added + real):.1%})")
    if dates[c][MONTH_YEAR]:
        print(f"    month + year without day: {dates[c][MONTH_YEAR]} (currently no date in the target)")

    print("\n[3] NUMBERS  (target: integer / decimal; invalid values are dropped)")
    for c in cfg["numbers"]:
        show(f"column '{c}'", nums[c], ok, n_ex[c])

    print("\n[4] ISBN  (identifier for gold standard / overlap only, not used for matching)")
    for c in cfg["isbn"]:
        show(f"'{c}'" + (" (all entries of all books)" if name == "DBpedia" else ""),
             isbns[c], sum(isbns[c].values()) or 1, i_ex[c])

    print("\n[5] LISTS / AUTHORS")
    for c in cfg["lists"]:
        show(f"column '{c}'", lists[c], ok, l_ex[c])
    if cfg["authors"]:
        print(f"\n    authors with a role in brackets (kept as they are): {with_role} books")
        print(f"    most common roles: {roles.most_common(6)}")

    print("\n[6] MISSING VALUES (empty or absent) per field")
    for k in sorted(filled, key=lambda x: ok - filled[x], reverse=True):
        missing = ok - filled[k]
        if missing:
            print(f"    {k:<24}{missing:>8}{missing / max(ok, 1):>8.1%}")

    if extra:
        print("\n[7] DATASET-SPECIFIC CHECKS")
        for k, n in sorted(extra.items()):
            print(f"    {k:<62}{n:>8}{n / max(ok, 1):>8.1%}")

    print(f"\n[8] SPECIAL CHARACTERS\n    books with '&' in title/publisher/genre/author (-> &amp; in the XML): {with_amp}")


def dataset_checks(name, rec, extra):
    if name == "DBpedia":
        r, y = rec.get("release_date", ""), rec.get("year", "")
        if re.fullmatch(r"\d{4}(-\d\d-\d\d)?", r):
            extra["release_date usable, year " + ("agrees" if y == r[:4] else
                  "missing" if not y else "DIFFERS")] += 1
        elif not r:
            extra["release_date empty, year " + ("present" if y else "empty")] += 1
        else:
            extra["release_date unusable, year " + ("present (fallback possible)" if y else "empty")] += 1
        entries = rec.get("isbns", [])
        extra[f"ISBN entries per book: {len(entries) if len(entries) < 3 else '3+'}"] += 1
        extra["entries with <= 9 characters (junk like 0, 978)"] += sum(len(x) <= 9 for x in entries)
        extra["books with at least one valid ISBN"] += any(isbn_class(x).startswith("valid") for x in entries)
    if name == "Amazon":
        a13, a10 = rec.get("isbn13", "").strip(), rec.get("isbn10", "").strip()
        extra["isbn13 empty, isbn10 present (fallback needed)"] += (not a13 and bool(a10))
        extra["isbn13 and isbn10 both empty"] += (not a13 and not a10)
        extra["isbn10 holds 13 digits (actually an ISBN-13)"] += bool(re.fullmatch(r"\d{13}", a10))


What the script checks, and why:

| Check | What we look at | Why it matters |
| --- | --- | --- |
| Structure | rows with the wrong number of columns | values sit in the wrong column and would break the mapping |
| Dates | year only, full date, month + year, real 1 January, not a date | the target is `xs:date`, and `parse-date` aborts the whole mapping on a value it cannot read |
| Is `-01-01` unique? | real 1 January vs. `-01-01` that we add to year-only values | if both exist, we can no longer tell them apart in the target |
| Numbers | valid, empty, invalid, `N page(s)` | MapForce aborts when it writes a bad value into a number field |
| ISBN | valid ISBN-10/13, placeholder, Excel damage, ASIN, junk | needed for the gold standard and the overlap estimate |
| Lists | format of `genres`, `awards`, `authors` | the separator decides how a list is split into single entries |
| Missing values | empty fields per column | shows how complete the target can be |
| Special characters | books containing `&` | explains `&amp;` in the XML output |

## 1. Goodreads (CSV)

A CSV with 25 columns, scraped from the web, so it is messy. We look at `publishDate` (many formats), the number columns `price`, `pages`, `rating` and `numRatings`, the ISBN, and the lists `genres` and `awards`. Authors are one comma-separated string with roles in brackets, e.g. `Suzanne Collins (Goodreads Author)`.

In [ ]:
analyse("Goodreads", CFG["Goodreads"], PATHS["Goodreads"])

**Results and what we did about them (Goodreads)**

*Dates.* `publishDate` comes in several formats: `MM/DD/YY` (e.g. `02/01/01`), year only (`2003`) and month name with an ordinal (`April 27th 2010`). Of about 52,500 books, 42,913 (81.8 %) have a full date, 4,362 (8.3 %) only a year, 1,219 (2.3 %) a real 1 January and 880 (1.7 %) are empty. 3,104 (5.9 %) match no known format. These include month + year such as `December 2002`, and six records where a list name stands in the date column instead of a date (records 8818, 9493, 18804, 20480, 21964, 33889), e.g. `Great Erotica`. That looks like a scraping artefact.

In MapForce, `Parse_Date` recognises each format with `match-pattern` and converts it to `YYYY-MM-DD`. Endings like `th` are cut off first because `parse-date` cannot read them. Two-digit years up to 26 become 20xx, the rest 19xx (the cut-off at 26 is our own assumption). Year-only values become `YYYY-01-01` because `xs:date` needs a full date. Everything else gives no date, so the element is missing in the target.

*Is `-01-01` unique?* No. Of the 5,581 dates ending in `-01-01`, 4,362 were added by us and 1,219 (about 22 %) are real. We cannot tell them apart in the target, so in Phase II only the year will be compared.

*Numbers.* `price` contains invalid values such as `1.189.88`, and `pages` sometimes holds the text `1 page` (23 times). `Safe_Number` returns nothing for an invalid number, so the element is left out. `Page_Count` turns `1 page` into `1` first.

*ISBN.* The column has the placeholder `9999999999999`, values damaged by Excel (`9.78044E+12`) and ASINs (`B0064CPN7I`). We only keep valid ISBN-10 and ISBN-13. `isbn` is `xs:string` in the target schema, otherwise MapForce tries to convert it to a number and aborts.

*Lists.* `genres` and `awards` are Python-style lists (`['A', 'B']`). `Clean_List` splits them at `', '`. Empty lists `[]` produce no elements. Roles in brackets after author names stay as they are.

## 2. DBpedia (XML)

An XML file with one `book` element per book. The lists (`authors`, `genres`, `isbns`) are already repeated child elements, and many elements are empty. We check `release_date` and `year`, `pages`, the ISBN entries and whether `year` is usable as a fallback when `release_date` is not.

In [ ]:
analyse("DBpedia", CFG["DBpedia"], PATHS["DBpedia"])

**Results and what we did about them (DBpedia)**

*Dates.* Of 59,270 books, `release_date` is year only for 34,355 (58.0 %), a full date for 12,925 (21.8 %), empty for 5,464 (9.2 %), month + year for 4,050 (6.8 %), not a date for 2,155 (3.6 %, e.g. `--11-01`, `20072010`, free text) and a real 1 January for 321 (0.5 %). Wherever `release_date` and `year` are both usable, the year is the same. We found no case where they differ. For 4,933 books `release_date` is unusable but `year` is filled. If `release_date` is empty, `year` is empty too.

In MapForce a conditional (if-else with `match-pattern`) checks whether `release_date` is a full date `YYYY-MM-DD` and parses it. Otherwise we take `year`, which gives us a date back for those 4,933 books.

*Is `-01-01` unique?* Even less than in Goodreads. Of 34,676 dates ending in `-01-01`, 34,355 (99.1 %) are year-only values we completed, and only 321 are real.

*ISBN.* Each book has several `isbn` entries and about every second entry is junk (`0`, `978`). 39 % of the books have no ISBN at all. We filter out entries of 9 characters or fewer and take the first remaining one with `first-items`. Hyphens (`978-0-8050-1334-4`) can be removed with `translate`.

*Numbers and lists.* `pages` is a number or empty (38.4 % missing), there is no `N page` text. `genres` is missing for 36.4 %. Lists are real XML elements and map directly to `author` and `genre`. `uri` is not mapped because it is not in the integrated schema.

## 3. Amazon (CSV)

Also a CSV, but with its own formats: `publication_date` as `April 30, 2019`, `authors` and `genres` as `["A","B"]`, and two ISBN columns (`isbn13`, `isbn10`). We check date formats, numbers (`pages`, `average_rating`, `ratings_count`, `price`), the list format and the ISBN gaps.

In [ ]:
analyse("Amazon", CFG["Amazon"], PATHS["Amazon"])

**Results and what we did about them (Amazon)**

*Dates.* 40,000 books, no shifted rows. 31,012 (77.5 %) have a full date like `April 30, 2019`, 7,022 (17.6 %) have 1 January, 1,966 (4.9 %) are empty. There are no year-only values, so nothing is completed here. The share of 1 January is high, probably Amazon fills in 1 January when the day is unknown. In MapForce, `parse-date` with the format `[MNn] [D1], [Y0001]` gives `2019-04-30`, guarded by `match-pattern`.

*Numbers.* There are no invalid values. `pages` is valid for 28,197 books (70.5 %) and empty for 11,803 (29.5 %), `price` valid for 35,276 (88.2 %) and empty for 4,724 (11.8 %), `average_rating` and `ratings_count` are valid for all 40,000. We still run them through `Safe_Number` so the three mappings work the same way.

*Lists.* `authors` is a JSON-style list for all 40,000 books. `genres` is one for 37,042 (92.6 %) and `[]` for 2,958 (7.4 %). There is no space after the comma, so `Clean_List` for Goodreads would not work. We cut off the outer characters with `substring` and split at `","` with `tokenize`. `[]` gives no `genre` elements.

*ISBN.* We use `isbn13` if it is filled and `isbn10` otherwise, because ISBN-13 is the current standard. `isbn13` is empty for 85 books (0.2 %), and there `isbn10` is always filled. In 3,029 books (7.6 %) the `isbn10` column actually holds 13 digits. This does not matter since `isbn13` has priority. There are also 6 ASINs, which are not ISBNs.

*Special characters.* 34,187 books (85 %) contain an `&` (e.g. `Literature & Fiction`). It shows up as `&amp;` in the output, which is normal XML escaping.

## 4. Comparison

| | Goodreads | DBpedia | Amazon |
| --- | --- | --- | --- |
| Format | CSV | XML | CSV |
| Books | about 52,500 | 59,270 | 40,000 |
| Date field | `publishDate` | `release_date` (fallback `year`) | `publication_date` |
| Real 1 January | 1,219 | 321 | 7,022 |
| `-01-01` added by us | 4,362 | 34,355 | 0 |
| ISBN source | `isbn` (placeholders, Excel damage, ASINs) | `isbns` (several, much junk) | `isbn13`, fallback `isbn10` |
| List format | Python list `['A', 'B']` | real XML elements | JSON list `["A","B"]` |
| Invalid numbers | yes (`1.189.88`, `1 page`) | no | no |

Because of the `-01-01` problem, only the year should be compared in the Phase II matching. Missing values (e.g. `page_count` in DBpedia, `series` in Amazon) are gaps in the source. Some missing values are caused by our own cleaning, for example invalid prices or unreadable dates that we drop.

## 5. What we did in MapForce

For each dataset there is one mapping (`.mfd`) and one output file (`goodreads_target.xml`, `dbpedia_target.xml`, `amazon_target.xml`). The target schema `target.xsd` has fixed types (`isbn` as `xs:string`, all elements optional), so MapForce does not guess types from an example file.

General rules for all three mappings:
- Source fields stay strings. Numbers and dates are never connected directly but go through a guarded function, because MapForce aborts the whole mapping on one bad value.
- CSV: `Rows` to `book`. XML: `book` to `book`. Lists go to the repeating child element (`author`, `genre`, `award`), not to the container.
- `id` is a prefix plus `auto-number` (`goodreads_`, `dbpedia_`, `amazon_`).
- Invalid values are dropped, so the element is missing in the output. We only fix a value when the fix is unambiguous (`1 page` to `1`).

Functions we built:
- `Parse_Date`: Goodreads dates, every format checked with `match-pattern`
- `Safe_Number`: text to number, nothing if invalid
- `Page_Count`: `N page(s)` to `N`, otherwise like `Safe_Number`
- `Clean_List`, `Clean_List_JSON`: split Goodreads and Amazon lists
- `Amazon_Date`: `Month D, YYYY`
- DBpedia: conditional on `release_date`, fallback `year`

After each mapping we check a few books in the output file: one with a year-only date, one with `1 page`, one with `&`, and one with an invalid value (the element must be missing).

## 6. Assumptions and limitations

- Two-digit years: up to 26 become 20xx, the rest 19xx. The cut-off is our assumption.
- Year-only values become `YYYY-01-01`. Month + year without a day currently gets no date, we have not decided yet whether to add day 01.
- Invalid numbers, ISBNs and dates are dropped, not guessed.
- We do not pre-clean the CSV files. The mappings read the original files, and all cleaning happens inside MapForce. The structure check only counts rows with the wrong number of columns (section [1] of each output), it does not remove them. Rows with a valid structure but a wrong value in one column (the six Goodreads date cases) stay in as books without a date.
- Author roles such as `(Illustrator)` are kept.
- `&` appears as `&amp;` in the output. That is normal XML and every parser reads it back as `&`.
- The Goodreads CSV must not be opened and saved in Excel, that damages the ISBNs.